# FATMOSS temporal turbulence

This tutorial generates a deterministic frozen-flow sequence, validates spatial statistics separately from temporal transport, and inserts the model into a Fiatlux optical element. Clone `EjjeSynho/FATMOSS` and add its source directory to `PYTHONPATH` before starting Jupyter.

In [ ]:
import matplotlib.pyplot as plt
import torch
from fiatlux import (Atmosphere, FatmossAtmosphereModel, FrozenFlowLayer, Grid,
    PhotometricBand, PlaneWave, Spectrum, estimate_translation,
    spatial_periodogram, spatial_structure_function, temporal_autocorrelation)

## Deterministic two-layer profile

FATMOSS weights multiply OPD amplitudes linearly. Normalization below turns the relative values 3:7 into 0.3 and 0.7.

In [ ]:
grid = Grid(135, 135, 0.04, 0.04)  # compatible with three FATMOSS cascades
dt = 2e-3
layers = [
    FrozenFlowLayer(0.15, 25.0, 5.0, 0.0, weight=3, altitude=0),
    FrozenFlowLayer(0.15, 25.0, 12.0, 70.0, weight=7, altitude=9000),
]
model = FatmossAtmosphereModel.create_frozen_flow(
    grid, layers, time_step=dt, batch_size=32, seed=42, normalize_weights=True)
opd = model.sequence_opd(64, advance=False)
print(opd.shape, opd.dtype, opd.device, f'{opd.std() * 1e9:.1f} nm RMS')

## Spatial validation

For metre-valued OPD, the 2-D PSD is in m⁴ and the structure function is in m². Neither diagnostic tests wind transport.

In [ ]:
fx, fy, psd = spatial_periodogram(opd, dx=grid.dx, dy=grid.dy)
rho, structure = spatial_structure_function(opd, spacing=grid.dx, max_lag=24)
frequency = torch.sqrt(fx.square() + fy.square())
valid = frequency > 0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(opd[0].cpu() * 1e9, origin='lower', cmap='RdBu_r')
axes[0].set_title('OPD at t=0 [nm]')
axes[1].loglog(frequency[valid].cpu(), psd[valid].cpu(), '.', alpha=.15)
axes[1].set(xlabel='Spatial frequency [cycles/m]', ylabel='OPD PSD [m⁴]')
axes[2].loglog(rho.cpu(), structure.cpu())
axes[2].set(xlabel='Separation [m]', ylabel='Structure function [m²]')
plt.tight_layout()

## Temporal validation

Autocorrelation measures decorrelation. Cross-correlation measures net transport. For a quantitative wind check, use one layer because a multilayer screen superposes several velocity vectors.

In [ ]:
correlation = temporal_autocorrelation(opd, max_lag=24)
lag = 5
shift_y, shift_x = estimate_translation(opd[0], opd[lag], dx=grid.dx, dy=grid.dy)
print(f'Shift after {lag * dt * 1e3:.0f} ms: dy={shift_y:.3f} m, dx={shift_x:.3f} m')
plt.plot(torch.arange(len(correlation)) * dt, correlation.cpu(), 'o-')
plt.xlabel('Lag [s]'); plt.ylabel('Normalized autocorrelation'); plt.grid(True)

## Optical and closed-loop cadence

Repeated optical evaluations stay at one atmospheric instant. Advance exactly once after each complete WFS/DM iteration; never advance during interaction-matrix calibration, and reset before the science loop.

In [ ]:
spectrum = Spectrum(magnitude=0, band=PhotometricBand.H, samples=1)
source = PlaneWave(spectrum)
atmosphere = Atmosphere(grid, model, recompute=True)
model.reset()
field_t0_a = atmosphere.apply(source.generate_field(grid))
field_t0_b = atmosphere.apply(source.generate_field(grid))
torch.testing.assert_close(field_t0_a.complex_amplitude, field_t0_b.complex_amplitude)
model.advance()  # once per completed AO iteration
field_t1 = atmosphere.apply(source.generate_field(grid))
assert not torch.equal(field_t0_a.complex_amplitude, field_t1.complex_amplitude)
print('Stable inside an iteration; changed at the explicit cadence.')

### Closed-loop pattern

```python
model.reset()
for _ in range(n_iterations):
    wfs_image = acquire_wfs_image()
    update_dm(wfs_image)
    model.advance()
```